In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
from delta.tables import DeltaTable
from pyspark.sql.functions import split, regexp_replace, col, lit, expr



########################################
########--TRANSFORMATIONS----##########
########################################

def transform_customer(df):
     df = df.withColumn("customer_name", split(col("customer_name",), ","))\
          .withColumn("First_Name", expr("get(customer_name, 1)"))\
          .withColumn("Last_Name", expr("get(customer_name, 0)"))\
          .withColumn("postcode", regexp_replace(col("postcode"), r"\.0$",""))\
          .withColumn("country", lit("USA"))\
          .drop("customer_name", "file_path",)

     df =df.dropDuplicates(["customer_id"])


     df = df.select('customer_id','first_name',
     'last_name',
     'tax_id',
     'tax_code',
     'state',
     'city',
     'postcode',
     'street',
     'number',
     'unit',
     'region',
     'district',
     'country',
     'lon',
     'lat',
     'ship_to_address',
     'valid_from',
     'valid_to',
     'units_purchased',
     'loyalty_segment',
     'last_update_ts'
     )
     return df


########################################
########--SCD-1 IMPLEMENTATION----######
########################################

target_table = "ecommerce_analytics.silver.customers"   

def scd_merge_table(spark, source_table, target_table, business_key):

     if not spark.catalog.tableExists("target_table"): 
          print("first Load: Creating Silver Table", target_table)
          source_table.write.format("delta").mode("overwrite").saveAsTable("target_table")
          print("Table Creatrd")

     else:
          print("incremental_load: performing SCD Type 1 Merge")

          delta_table = DeltaTable.forName(spark, "target_table")

          merge_condition = " AND ".join(
               [f"target.{col} = source.{col}" for col in business_key]
          )
         

          delta_table.alias("target").merge(source_table.alias("source"),
                                             merge_condition
                                             ).whenMatchedUpdateAll()\
                                             .whenNotMatchedInsertAll()\
                                             .execute()
          print ("Merge Succefully Complited")


########################################
########--MAIN LOGIC----##########
########################################

source_table = "ecommerce_analytics.bronze.customers"
target_table = "ecommerce_analytics.silver.customers"
business_key = ["customer_id"] 
df = spark.read.table(source_table)

source_table = transform_customer(df)
scd_merge_table(spark, source_table, target_table, business_key)





